# 🎓 Mini-Projets TMA 2025-2026 — SupNum
### Dr. EL BENANY Med Mahmoud

---

Ce notebook contient **tous les défis** du mini-projet TMA, niveau débutant.
Chaque section est indépendante, commentée et expliquée pas à pas.

**Table des matières :**
- 🔴 **DÉFI OBLIGATOIRE** — Capture & Analyse Spectrale de la Voix
- 🟠 **Défi 1** — L'Archéologue Acoustique (NASA/JFK)
- 🟡 **Défi 2** — Le Magicien de la Séparation (BSS / ICA)
- 🟢 **Défi 3** — Le Détective de l'Aliasing
- 🔵 **Défi 4** — Le Diagnostic du Futur (ECG)
- 🟣 **Défi 5** — L'Opticien Numérique (Déconvolution)
- 🩷 **Défi 6** — Le Chasseur de Fantômes (Bruit ISO)
- 🤍 **Défi 7** — L'Espion Fréquentiel (Stéganographie)
- 🖤 **Défi 8** — Le Compresseur Intelligent (MP3 Light)
- 🟫 **Défi 9** — L'Annulateur d'Écho
- 🔶 **Défi 10** — Le Radar Numérique


## 📦 Installation des bibliothèques nécessaires
Exécutez cette cellule en premier !

In [ ]:
# Installation de toutes les bibliothèques utilisées dans ce projet
pip install numpy scipy matplotlib librosa scikit-image Pillow -q

# Importation des bibliothèques de base
import numpy as np                     # Calcul numérique
import matplotlib.pyplot as plt        # Graphiques
import scipy.signal as signal          # Traitement du signal
import scipy.io.wavfile as wav         # Lecture de fichiers audio WAV
from scipy.fft import fft, fftfreq, ifft  # Transformée de Fourier
import warnings
warnings.filterwarnings('ignore')

print('✅ Toutes les bibliothèques sont prêtes !')

---
# 🔴 DÉFI OBLIGATOIRE — Capture & Analyse Spectrale de la Voix

**Objectif :** Enregistrer sa voix dans Colab, afficher le signal, calculer la FFT,
identifier la fréquence fondamentale (Pitch), et comparer les voix du binôme.

> 💡 **C'est quoi la FFT ?** La Transformée de Fourier Rapide décompose un son en
> ses fréquences constituantes — comme un prisme qui décompose la lumière en couleurs.
> La fréquence fondamentale (Pitch) est la note principale de votre voix.

In [ ]:
# ===== ÉTAPE 1 : Enregistrement de la voix via JavaScript dans Colab =====
# Ce code JavaScript interagit avec le navigateur pour accéder au microphone.
# Il enregistre ~5 secondes, puis envoie le résultat à Python.

from IPython.display import Javascript, display
from google.colab import output
import base64, io

RECORD_JS = """
async function record() {
  // Demander la permission au microphone
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);
  const chunks = [];

  recorder.ondataavailable = e => chunks.push(e.data);

  recorder.onstop = () => {
    const blob = new Blob(chunks, { type: 'audio/webm' });
    const reader = new FileReader();
    reader.onloadend = () => {
      // Envoyer les données encodées en base64 à Python
      const base64 = reader.result.split(',')[1];
      google.colab.kernel.invokeFunction('notebook.save_audio', [base64], {});
    };
    reader.readAsDataURL(blob);
  };

  // Enregistrer pendant 5 secondes
  recorder.start();
  document.getElementById('status').textContent = '🎙️ Enregistrement en cours... (5s)';
  await new Promise(r => setTimeout(r, 5000));
  recorder.stop();
  document.getElementById('status').textContent = '✅ Enregistrement terminé !';
}

// Créer un bouton et une zone de statut
const btn = document.createElement('button');
btn.textContent = '🎤 Cliquez pour enregistrer votre voix (5s)';
btn.style = 'padding:10px 20px; font-size:16px; cursor:pointer; background:#4CAF50; color:white; border:none; border-radius:5px';
btn.onclick = record;
const status = document.createElement('p');
status.id = 'status';
status.textContent = '⏳ En attente du clic...';
document.body.appendChild(btn);
document.body.appendChild(status);
"""

audio_data_global = {}

def save_audio(base64_audio):
    """Fonction Python appelée par JavaScript pour sauvegarder l'audio."""
    audio_bytes = base64.b64decode(base64_audio)
    with open('/content/ma_voix.webm', 'wb') as f:
        f.write(audio_bytes)
    audio_data_global['saved'] = True
    print('✅ Voix sauvegardée dans /content/ma_voix.webm')

output.register_callback('notebook.save_audio', save_audio)
display(Javascript(RECORD_JS))
print('👆 Cliquez sur le bouton vert ci-dessus pour commencer l\'enregistrement.')

In [ ]:
# ===== ÉTAPE 2 : Simulation d'une voix si enregistrement impossible =====
# Si le microphone ne fonctionne pas (ex: environnement restreint),
# on SIMULE une voix synthétique pour démontrer les calculs.

import os

def simuler_voix(freq_fondamentale=150, duree=5, fs=22050, nom='Voix A'):
    """
    Crée une voix simulée avec :
    - Une fréquence fondamentale (le 'Pitch', ex: 150 Hz pour un homme, 250 Hz pour une femme)
    - Des harmoniques (multiples de la fondamentale) qui donnent le timbre
    - Un peu de bruit pour simuler le son réel
    """
    t = np.linspace(0, duree, fs * duree)  # Axe du temps
    
    # La voix = fréquence fondamentale + harmoniques (2f, 3f, 4f...)
    voix = np.sin(2 * np.pi * freq_fondamentale * t)          # Fondamentale (amplitude 1)
    voix += 0.5 * np.sin(2 * np.pi * 2 * freq_fondamentale * t)  # 2ème harmonique
    voix += 0.3 * np.sin(2 * np.pi * 3 * freq_fondamentale * t)  # 3ème harmonique
    voix += 0.2 * np.sin(2 * np.pi * 4 * freq_fondamentale * t)  # 4ème harmonique
    voix += 0.1 * np.random.randn(len(t))                         # Bruit léger
    
    # Normaliser entre -1 et 1
    voix = voix / np.max(np.abs(voix))
    
    print(f'✅ {nom} simulée : freq fondamentale = {freq_fondamentale} Hz')
    return voix, t, fs

# Simuler deux voix (comme si c'était deux membres du binôme)
voix_A, t_A, fs = simuler_voix(freq_fondamentale=130, nom='Étudiant A (voix grave, homme)')
voix_B, t_B, _  = simuler_voix(freq_fondamentale=240, nom='Étudiant B (voix aiguë, femme)')

print('\n📌 Explication : La fréquence fondamentale d\'un homme est typiquement 85-180 Hz,')
print('   celle d\'une femme 165-255 Hz. C\'est ce qui détermine si la voix est grave ou aiguë.')

In [ ]:
# ===== ÉTAPE 3 : Affichage du signal temporel =====
# Le signal temporel montre l'amplitude (volume) du son au fil du temps.

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
fig.suptitle('📊 DÉFI OBLIGATOIRE — Signaux temporels des deux voix', fontsize=14, fontweight='bold')

# Afficher seulement les 0.05 premières secondes (pour voir les oscillations)
nb_points = int(0.05 * fs)

axes[0].plot(t_A[:nb_points], voix_A[:nb_points], color='blue', linewidth=0.8)
axes[0].set_title('🎵 Étudiant A (130 Hz — voix grave)', fontsize=12)
axes[0].set_xlabel('Temps (secondes)')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_B[:nb_points], voix_B[:nb_points], color='red', linewidth=0.8)
axes[1].set_title('🎵 Étudiant B (240 Hz — voix aiguë)', fontsize=12)
axes[1].set_xlabel('Temps (secondes)')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/defi0_signaux_temporels.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Graphique sauvegardé.')

In [ ]:
# ===== ÉTAPE 4 : Calcul et visualisation de la FFT =====
# La FFT révèle QUELLES fréquences composent le son.

def calculer_fft(signal_audio, fs):
    """
    Calcule la FFT d'un signal audio et retourne les fréquences et amplitudes.
    On ne garde que les fréquences positives (la FFT est symétrique).
    """
    N = len(signal_audio)                       # Nombre de points
    spectre = np.abs(fft(signal_audio))[:N//2]  # Amplitudes (moitié positive)
    freqs = fftfreq(N, d=1/fs)[:N//2]           # Fréquences correspondantes (Hz)
    return freqs, spectre

freqs_A, spectre_A = calculer_fft(voix_A, fs)
freqs_B, spectre_B = calculer_fft(voix_B, fs)

# Trouver la fréquence fondamentale (le pic le plus haut)
# On cherche dans la plage 50-500 Hz (plage vocale)
masque = (freqs_A > 50) & (freqs_A < 500)
f0_A = freqs_A[masque][np.argmax(spectre_A[masque])]
f0_B = freqs_B[masque][np.argmax(spectre_B[masque])]

print(f'🎵 Fréquence fondamentale (Pitch) :')
print(f'   Étudiant A : {f0_A:.1f} Hz  → Voix {"grave" if f0_A < 180 else "aiguë"}')
print(f'   Étudiant B : {f0_B:.1f} Hz  → Voix {"grave" if f0_B < 180 else "aiguë"}')
print(f'   👑 Voix la plus grave : {"A" if f0_A < f0_B else "B"}')

In [ ]:
# ===== ÉTAPE 5 : Superposition des spectres (graphique comparatif) =====

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('📊 DÉFI OBLIGATOIRE — Spectres FFT des deux voix', fontsize=14, fontweight='bold')

# Spectre individuel
limite_freq = 2000  # On affiche jusqu'à 2000 Hz (la voix est principalement là)
masq = freqs_A < limite_freq

axes[0].plot(freqs_A[masq], spectre_A[masq], color='blue', label='Étudiant A (130 Hz)')
axes[0].plot(freqs_B[masq], spectre_B[masq], color='red', alpha=0.7, label='Étudiant B (240 Hz)')
axes[0].axvline(f0_A, color='blue', linestyle='--', alpha=0.5, label=f'Pitch A = {f0_A:.0f} Hz')
axes[0].axvline(f0_B, color='red', linestyle='--', alpha=0.5, label=f'Pitch B = {f0_B:.0f} Hz')
axes[0].set_title('Superposition des spectres fréquentiels', fontsize=12)
axes[0].set_xlabel('Fréquence (Hz)')
axes[0].set_ylabel('Amplitude')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Annotation des harmoniques de A
axes[0].annotate('Fondamentale A', xy=(f0_A, spectre_A[np.argmin(np.abs(freqs_A - f0_A))]),
                xytext=(f0_A + 100, spectre_A[np.argmin(np.abs(freqs_A - f0_A))]),
                arrowprops=dict(arrowstyle='->', color='blue'), color='blue', fontsize=9)

# Zoom sur les harmoniques
axes[1].plot(freqs_A[masq], spectre_A[masq], color='blue', alpha=0.8, label='Étudiant A')
axes[1].plot(freqs_B[masq], spectre_B[masq], color='red', alpha=0.8, label='Étudiant B')

# Marquer les harmoniques (2f0, 3f0, 4f0)
for i, (f0, couleur, nom) in enumerate([(f0_A, 'blue', 'A'), (f0_B, 'red', 'B')]):
    for k in range(1, 5):
        axes[1].axvline(k * f0, color=couleur, linestyle=':', alpha=0.4)
        if k * f0 < limite_freq:
            axes[1].text(k * f0, axes[1].get_ylim()[1] * 0.9 if i == 0 else axes[1].get_ylim()[1] * 0.7,
                       f'{k}f₀{nom}', fontsize=7, color=couleur, ha='center')

axes[1].set_title('Harmoniques des deux voix (1f₀, 2f₀, 3f₀, 4f₀...)', fontsize=12)
axes[1].set_xlabel('Fréquence (Hz)')
axes[1].set_ylabel('Amplitude')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/defi0_spectres_fft.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Graphique sauvegardé.')

In [ ]:
# ===== ÉTAPE 6 & 7 : Analyse et Synthèse =====

print('=' * 60)
print('📋 ANALYSE ET SYNTHÈSE — DÉFI OBLIGATOIRE')
print('=' * 60)
print(f'''
1. FRÉQUENCES FONDAMENTALES :
   - Étudiant A : {f0_A:.1f} Hz (voix grave)
   - Étudiant B : {f0_B:.1f} Hz (voix aiguë)
   - Différence  : {abs(f0_B - f0_A):.1f} Hz

2. QUI A LA VOIX LA PLUS GRAVE ?
   → Étudiant {"A" if f0_A < f0_B else "B"} (fréquence fondamentale plus basse)

3. QUI A LE SPECTRE LE PLUS RICHE EN HAUTES FRÉQUENCES ?
   → On compare l'énergie spectrale au-delà de 1000 Hz.
''')

# Comparer l'énergie haute fréquence (>1000 Hz)
masq_hf = freqs_A > 1000
energie_hf_A = np.sum(spectre_A[masq_hf]**2)
energie_hf_B = np.sum(spectre_B[masq_hf]**2)
print(f'   Énergie haute fréquence A : {energie_hf_A:.0f}')
print(f'   Énergie haute fréquence B : {energie_hf_B:.0f}')
print(f'   → Étudiant {"A" if energie_hf_A > energie_hf_B else "B"} a le spectre le plus riche en hautes fréquences.')

print('''
4. DIFFICULTÉ DE SÉPARATION (lien avec Défi 2) :
   Si les deux voix sont mélangées en un seul signal, les séparer est difficile car :
   - Leurs spectres se CHEVAUCHENT dans certaines zones de fréquence.
   - Un filtre simple ne suffira pas à isoler chaque voix.
   - Il faudra utiliser des techniques avancées (ICA) — voir Défi 2.
''')
print('=' * 60)

---
# 🟠 DÉFI 1 — L'Archéologue Acoustique (NASA/JFK)

**Contexte :** Un discours historique est dégradé par du bruit blanc et un sifflement à 1000 Hz.  
**Mission :** Restaurer le signal audio en éliminant les parasites.

> 💡 **Concepts clés :**
> - **Bruit blanc** = bruit contenant toutes les fréquences (comme la neige sur un vieux téléviseur)
> - **Filtre Notch** = filtre qui supprime UNE fréquence précise sans toucher les autres
> - **SNR (Signal/Noise Ratio)** = mesure de la qualité du signal (plus c'est élevé, mieux c'est)

In [ ]:
# ===== DÉFI 1 : Étape 1 — Créer un signal dégradé =====
# On simule le discours de Kennedy dégradé :
# Signal = voix propre + bruit blanc + sifflement à 1000 Hz

print('🎙️ DÉFI 1 — L\'Archéologue Acoustique')
print('-' * 50)

# Paramètres
fs = 22050          # Fréquence d'échantillonnage (Hz) — nombre de mesures par seconde
duree = 5           # Durée du signal (secondes)
t = np.linspace(0, duree, fs * duree)  # Axe temporel

# 1. Simuler la "voix de Kennedy" — un mélange de fréquences vocales
voix_originale = (
    0.8 * np.sin(2 * np.pi * 150 * t) +  # Fondamentale (voix grave)
    0.5 * np.sin(2 * np.pi * 300 * t) +  # 1er harmonique
    0.3 * np.sin(2 * np.pi * 450 * t) +  # 2ème harmonique
    0.2 * np.sin(2 * np.pi * 600 * t)    # 3ème harmonique
)

# 2. Ajouter le bruit blanc (magnétique)
np.random.seed(42)  # Pour avoir des résultats reproductibles
bruit_blanc = 0.4 * np.random.randn(len(t))

# 3. Ajouter le sifflement électrique à 1000 Hz
sifflement = 0.6 * np.sin(2 * np.pi * 1000 * t)

# 4. Signal dégradé = voix + bruit + sifflement
signal_degrade = voix_originale + bruit_blanc + sifflement

print('✅ Signal dégradé créé :')
print(f'   - Voix originale : fréquences 150-600 Hz')
print(f'   - Bruit blanc ajouté (amplitude 0.4)')
print(f'   - Sifflement parasite à 1000 Hz (amplitude 0.6)')

In [ ]:
# ===== DÉFI 1 : Étape 2 — Identifier le sifflement sur le spectrogramme =====

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('DÉFI 1 — Identification du sifflement parasite à 1000 Hz', fontsize=14, fontweight='bold')

# Spectrogramme du signal dégradé
# Un spectrogramme = FFT calculée sur des petites fenêtres de temps (vue 2D : temps x fréquence)
f_spec, t_spec, Sxx = signal.spectrogram(signal_degrade, fs, nperseg=512)

axes[0].pcolormesh(t_spec, f_spec[:200], 10*np.log10(Sxx[:200]+1e-10), shading='auto', cmap='inferno')
axes[0].set_title('Spectrogramme (signal dégradé)\n→ Ligne rouge = sifflement 1000 Hz')
axes[0].set_xlabel('Temps (s)')
axes[0].set_ylabel('Fréquence (Hz)')
axes[0].axhline(y=1000, color='cyan', linestyle='--', linewidth=2, label='1000 Hz')
axes[0].legend()

# Spectre FFT du signal dégradé
N = len(signal_degrade)
freqs = fftfreq(N, d=1/fs)[:N//2]
spectre_degrade = np.abs(fft(signal_degrade))[:N//2]
spectre_original = np.abs(fft(voix_originale))[:N//2]

axes[1].plot(freqs[freqs < 2000], spectre_degrade[freqs < 2000], color='red', label='Signal dégradé')
axes[1].plot(freqs[freqs < 2000], spectre_original[freqs < 2000], color='blue', alpha=0.7, label='Voix originale')
axes[1].axvline(1000, color='orange', linestyle='--', linewidth=2, label='Raie 1000 Hz')
axes[1].set_title('Spectre FFT\n→ Pic orange = parasite à 1000 Hz')
axes[1].set_xlabel('Fréquence (Hz)')
axes[1].set_ylabel('Amplitude')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Signal temporel
nb = int(0.02 * fs)
axes[2].plot(t[:nb], signal_degrade[:nb], color='red', alpha=0.7, label='Dégradé')
axes[2].plot(t[:nb], voix_originale[:nb], color='blue', label='Original')
axes[2].set_title('Signal temporel\n(zoom 20ms)')
axes[2].set_xlabel('Temps (s)')
axes[2].set_ylabel('Amplitude')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/defi1_identification_bruit.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ===== DÉFI 1 : Étape 3 — Filtre Notch (réjecteur) à 1000 Hz =====
# Le filtre Notch supprime UNIQUEMENT la fréquence de 1000 Hz.
# Paramètre Q : plus Q est grand, plus le filtre est sélectif (bande étroite).

print('🔧 Application du Filtre Notch à 1000 Hz...')

# Conception du filtre Notch avec scipy
freq_notch = 1000  # Fréquence à supprimer (Hz)
Q = 30             # Facteur de qualité (sélectivité du filtre)

# iirnotch retourne les coefficients b, a du filtre numérique
b_notch, a_notch = signal.iirnotch(freq_notch, Q, fs)

# Application du filtre au signal dégradé
signal_apres_notch = signal.filtfilt(b_notch, a_notch, signal_degrade)
# filtfilt applique le filtre en avant ET en arrière (pas de déphasage)

print(f'✅ Filtre Notch appliqué : fréquence = {freq_notch} Hz, Q = {Q}')
print('   → Le sifflement à 1000 Hz est éliminé !')

# ===== ÉTAPE 4 : Soustraction Spectrale pour le bruit blanc =====
# On estime le bruit à partir d'une zone de silence (les 0.5 premières secondes)
# puis on le soustrait du spectre du signal.

print('\n🔧 Application de la Soustraction Spectrale pour le bruit blanc...')

def soustraction_spectrale(signal_bruit, fs, duree_silence=0.5, facteur=1.5):
    """
    Soustraction spectrale :
    1. Estimer le spectre du bruit sur une zone de silence
    2. Soustraire cette estimation de chaque trame du signal
    3. Reconstruire le signal propre
    """
    taille_trame = 1024   # Taille de chaque trame analysée
    chevauchement = 512   # Chevauchement entre trames
    
    # Estimer le profil de bruit sur la zone de silence initiale
    nb_silence = int(duree_silence * fs)
    trame_silence = signal_bruit[:nb_silence]
    spectre_bruit = np.mean(np.abs(fft(trame_silence[:taille_trame])))
    
    # Traiter le signal trame par trame
    signal_propre = np.zeros_like(signal_bruit)
    
    for debut in range(0, len(signal_bruit) - taille_trame, chevauchement):
        fin = debut + taille_trame
        trame = signal_bruit[debut:fin]
        
        # Calcul FFT de la trame
        TRAME_FFT = fft(trame)
        amplitude = np.abs(TRAME_FFT)
        phase = np.angle(TRAME_FFT)
        
        # Soustraction du bruit (avec plancher à 0 pour éviter des valeurs négatives)
        amplitude_propre = np.maximum(amplitude - facteur * spectre_bruit, 0)
        
        # Reconstruction de la trame propre
        trame_propre = np.real(ifft(amplitude_propre * np.exp(1j * phase)))
        signal_propre[debut:fin] += trame_propre * np.hanning(taille_trame)
    
    return signal_propre

# Appliquer la soustraction spectrale
signal_restaure = soustraction_spectrale(signal_apres_notch, fs)
print('✅ Soustraction spectrale terminée !')

In [ ]:
# ===== DÉFI 1 : Étape 5 — Calcul du SNR et visualisation finale =====

def calculer_snr(signal_propre, signal_bruit):
    """SNR = 10 * log10(puissance signal / puissance bruit) en décibels (dB)."""
    puissance_signal = np.mean(signal_propre ** 2)
    bruit = signal_bruit - signal_propre
    puissance_bruit = np.mean(bruit ** 2) + 1e-10  # Éviter division par zéro
    return 10 * np.log10(puissance_signal / puissance_bruit)

# Aligner les longueurs
longueur_min = min(len(voix_originale), len(signal_restaure))
snr_avant = calculer_snr(voix_originale[:longueur_min], signal_degrade[:longueur_min])
snr_apres = calculer_snr(voix_originale[:longueur_min], signal_restaure[:longueur_min])

print(f'📊 RÉSULTATS — Qualité du signal :')
print(f'   SNR avant restauration : {snr_avant:.2f} dB')
print(f'   SNR après restauration : {snr_apres:.2f} dB')
print(f'   Gain de qualité        : +{snr_apres - snr_avant:.2f} dB 🎉')

# Visualisation comparative
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('DÉFI 1 — Restauration du signal audio (avant / après)', fontsize=14, fontweight='bold')

nb = int(0.05 * fs)
# Signaux temporels
axes[0,0].plot(t[:nb], signal_degrade[:nb], color='red', linewidth=0.8)
axes[0,0].set_title(f'Signal dégradé (SNR = {snr_avant:.1f} dB)')
axes[0,0].set_xlabel('Temps (s)'); axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(t[:nb], signal_restaure[:nb], color='green', linewidth=0.8)
axes[0,1].set_title(f'Signal restauré (SNR = {snr_apres:.1f} dB) ✅')
axes[0,1].set_xlabel('Temps (s)'); axes[0,1].grid(True, alpha=0.3)

# Spectres FFT
spectre_restaure = np.abs(fft(signal_restaure))[:N//2]
axes[1,0].plot(freqs[freqs < 2000], spectre_degrade[freqs < 2000], color='red', label='Dégradé')
axes[1,0].axvline(1000, color='orange', linestyle='--', label='Sifflement 1000 Hz')
axes[1,0].set_title('Spectre avant restauration\n(sifflement visible à 1000 Hz)')
axes[1,0].set_xlabel('Fréquence (Hz)'); axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(freqs[freqs < 2000], spectre_restaure[freqs < 2000], color='green', label='Restauré')
axes[1,1].plot(freqs[freqs < 2000], spectre_original[freqs < 2000], color='blue', alpha=0.5, label='Original')
axes[1,1].set_title('Spectre après restauration\n(sifflement supprimé ✅)')
axes[1,1].set_xlabel('Fréquence (Hz)'); axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/defi1_restauration.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Défi 1 terminé — Le discours a été "remasterisé" !')

---
# 🟡 DÉFI 2 — Le Magicien de la Séparation (BSS / ICA)

**Contexte :** Deux voix sont mélangées dans un seul micro.  
**Mission :** Séparer les deux voix sans aucune information préalable (Blind Source Separation).

> 💡 **Concepts clés :**
> - **ICA (Independent Component Analysis)** = algorithme qui cherche des signaux statistiquement indépendants
> - **SVD (Singular Value Decomposition)** = décomposition matricielle qui révèle la structure des données
> - **Problème du Cocktail Party** = séparer des voix dans un environnement bruyant

In [ ]:
# ===== DÉFI 2 : Étape 1 — Créer le mélange de deux voix =====

print('🎭 DÉFI 2 — Le Magicien de la Séparation (BSS / ICA)')
print('-' * 55)

# Simuler deux sources vocales distinctes
fs2 = 8000      # Fréquence d'échantillonnage réduite pour aller plus vite
duree2 = 3      # 3 secondes suffisent
N2 = fs2 * duree2
t2 = np.linspace(0, duree2, N2)

# Source 1 : Voix grave (type homme)
source1 = np.sin(2 * np.pi * 120 * t2) + 0.5 * np.sin(2 * np.pi * 240 * t2) + 0.3 * np.random.randn(N2)

# Source 2 : Voix aiguë (type femme) avec variation (parole)
source2 = (np.sin(2 * np.pi * 220 * t2) * (1 + 0.5 * np.sin(2 * np.pi * 3 * t2)) +
           0.3 * np.random.randn(N2))

# Normalisation
source1 /= np.std(source1)
source2 /= np.std(source2)

# Matrice de mélange (inconnue dans un cas réel — c'est le 'blind' de BSS)
# Ces coefficients simulent comment les sons se propagent dans la pièce
A = np.array([[0.7, 0.3],   # Micro 1 : 70% voix1 + 30% voix2
              [0.4, 0.6]])   # Micro 2 : 40% voix1 + 60% voix2

# Mélange = A * [source1, source2]
S = np.vstack([source1, source2])     # Sources empilées
X = A @ S                              # Signaux mélangés (observés par les micros)

micro1 = X[0]  # Ce que capte le micro 1
micro2 = X[1]  # Ce que capte le micro 2

print(f'✅ Simulation du mélange :')
print(f'   Source 1 (voix grave, 120 Hz) + Source 2 (voix aiguë, 220 Hz)')
print(f'   Matrice de mélange A :')
print(f'   {A}')
print(f'\n   Micro 1 entend : 70% voix A + 30% voix B')
print(f'   Micro 2 entend : 40% voix A + 60% voix B')

In [ ]:
# ===== DÉFI 2 : Étape 2 — SVD pour analyser la structure =====
# La SVD décompose la matrice X en X = U * Sigma * V^T
# Cela permet de voir combien de 'sources indépendantes' existent.

print('🔬 Analyse par SVD (Singular Value Decomposition)...')

U, sigma, Vt = np.linalg.svd(X, full_matrices=False)

print(f'Valeurs singulières : {sigma}')
print(f'→ Il y a {len(sigma)} composantes principales (sources)')

# Blanchiment du signal (Whitening) — préparation pour ICA
# Le blanchiment rend les signaux décorrélés et de variance unitaire
print('\n🔧 Blanchiment du signal (Whitening)...')

# Centrer les signaux (soustraire la moyenne)
X_centre = X - X.mean(axis=1, keepdims=True)

# Matrice de covariance
cov = np.cov(X_centre)

# Décomposition en valeurs propres
valeurs_propres, vecteurs_propres = np.linalg.eigh(cov)
D = np.diag(1.0 / np.sqrt(valeurs_propres + 1e-10))
W_blanchi = D @ vecteurs_propres.T

# Signal blanchi
X_blanchi = W_blanchi @ X_centre
print('✅ Signal blanchi prêt pour ICA')

In [ ]:
# ===== DÉFI 2 : Étape 3 — Algorithme ICA (FastICA simplifié) =====
# L'ICA cherche des composantes INDÉPENDANTES en maximisant la non-gaussianité.
# On utilise la fonction tanh qui mesure la non-gaussianité.

print('🧠 Application de l\'algorithme FastICA...')

def fast_ica_simple(X_blanchi, nb_iterations=1000, tolerance=1e-5):
    """
    Implémentation simplifiée de FastICA :
    - Initialise un vecteur de séparation w aléatoire
    - Itère pour maximiser la non-gaussianité (via tanh)
    - Retourne les composantes indépendantes
    """
    n_sources, n_echantillons = X_blanchi.shape
    
    # Matrice de séparation W
    np.random.seed(0)
    W = np.random.randn(n_sources, n_sources)
    
    for i in range(nb_iterations):
        W_ancien = W.copy()
        
        # Projection
        Y = W @ X_blanchi
        
        # Mise à jour par gradient (règle de Newton)
        # g(y) = tanh(y) — approximation de la non-gaussianité
        g_Y = np.tanh(Y)
        g_prime_Y = 1 - g_Y ** 2
        
        W = (g_Y @ X_blanchi.T / n_echantillons - 
             g_prime_Y.mean(axis=1, keepdims=True) * W)
        
        # Orthogonalisation (décorrélation des composantes)
        U_ica, _, Vt_ica = np.linalg.svd(W)
        W = U_ica @ Vt_ica
        
        # Vérifier la convergence
        if np.max(np.abs(np.abs(np.diag(W @ W_ancien.T)) - 1)) < tolerance:
            print(f'   ✅ Convergence atteinte à l\'itération {i+1}')
            break
    
    return W @ X_blanchi

# Appliquer ICA
sources_separees = fast_ica_simple(X_blanchi)

# Normaliser les sources
sources_separees[0] /= np.std(sources_separees[0])
sources_separees[1] /= np.std(sources_separees[1])

print('\n✅ ICA terminée — Deux sources séparées !')

In [ ]:
# ===== DÉFI 2 : Étape 4 — Visualisation et évaluation =====

fig, axes = plt.subplots(3, 2, figsize=(16, 12))
fig.suptitle('DÉFI 2 — Séparation de sources (BSS / ICA)', fontsize=14, fontweight='bold')

nb2 = int(0.5 * fs2)  # Afficher 0.5 secondes

# Ligne 1 : Sources originales
axes[0,0].plot(t2[:nb2], source1[:nb2], color='blue')
axes[0,0].set_title('Source 1 originale (voix grave 120 Hz)')
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(t2[:nb2], source2[:nb2], color='red')
axes[0,1].set_title('Source 2 originale (voix aiguë 220 Hz)')
axes[0,1].grid(True, alpha=0.3)

# Ligne 2 : Signaux mélangés (ce que les micros entendent)
axes[1,0].plot(t2[:nb2], micro1[:nb2], color='purple')
axes[1,0].set_title('Micro 1 (mélange : 70% voix1 + 30% voix2)')
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(t2[:nb2], micro2[:nb2], color='orange')
axes[1,1].set_title('Micro 2 (mélange : 40% voix1 + 60% voix2)')
axes[1,1].grid(True, alpha=0.3)

# Ligne 3 : Sources séparées par ICA
axes[2,0].plot(t2[:nb2], sources_separees[0,:nb2], color='green')
axes[2,0].set_title('Source séparée par ICA n°1 ✅')
axes[2,0].grid(True, alpha=0.3)

axes[2,1].plot(t2[:nb2], sources_separees[1,:nb2], color='darkgreen')
axes[2,1].set_title('Source séparée par ICA n°2 ✅')
axes[2,1].grid(True, alpha=0.3)

for ax in axes.flat:
    ax.set_xlabel('Temps (s)')
    ax.set_ylabel('Amplitude')

plt.tight_layout()
plt.savefig('/content/defi2_separation_ica.png', dpi=150, bbox_inches='tight')
plt.show()

# Corrélation pour évaluer la qualité de séparation
corr1 = abs(np.corrcoef(source1, sources_separees[0])[0,1])
corr2 = abs(np.corrcoef(source2, sources_separees[1])[0,1])
print(f'\n📊 Qualité de la séparation :')
print(f'   Corrélation source1 ↔ séparée1 : {corr1:.3f}  (1.0 = parfait)')
print(f'   Corrélation source2 ↔ séparée2 : {corr2:.3f}  (1.0 = parfait)')
print('\n✅ Défi 2 terminé — Les voix ont été magiquement séparées !')

---
# 🟢 DÉFI 3 — Le Détective de l'Aliasing (Le Spectre Fantôme)

**Contexte :** Un capteur mal configuré échantillonne trop lentement → des fréquences fantômes apparaissent.  
**Mission :** Identifier et corriger l'aliasing.

> 💡 **Théorème de Shannon :** Pour capturer correctement un signal de fréquence max `f_max`,
> il faut échantillonner à au moins `2 × f_max` Hz. En dessous, l'aliasing apparaît.

In [ ]:
# ===== DÉFI 3 : Démonstration de l'aliasing =====

print('🕵️ DÉFI 3 — Le Détective de l\'Aliasing')
print('-' * 50)

# Signal original : sinusoïde à 300 Hz
f_signal = 300    # Fréquence du signal (Hz)
fs_correct = 8000  # Fréquence d'échantillonnage correcte (> 2 * 300 = 600 Hz)
fs_trop_bas = 500  # Fréquence d'échantillonnage incorrecte (< 2 * 300 = 600 Hz !)

duree3 = 0.1  # 100 ms suffisent pour illustrer

# Signal continu (haute résolution temporelle)
t_continu = np.linspace(0, duree3, 10000)
signal_continu = np.sin(2 * np.pi * f_signal * t_continu)

# Échantillonnage correct (fs_correct = 8000 Hz >> 2 * 300 = 600 Hz)
t_correct = np.arange(0, duree3, 1/fs_correct)
signal_correct = np.sin(2 * np.pi * f_signal * t_correct)

# Échantillonnage trop bas (fs_trop_bas = 500 Hz < 2 * 300 = 600 Hz → VIOLATION !)
t_alias = np.arange(0, duree3, 1/fs_trop_bas)
signal_alias = np.sin(2 * np.pi * f_signal * t_alias)

# Fréquence fantôme (aliasée) = f_signal - fs_trop_bas (repliement)
f_alias = abs(f_signal - fs_trop_bas)

print(f'Signal à capturer   : {f_signal} Hz')
print(f'Limite de Shannon   : {2*f_signal} Hz (doit échantillonner AU MOINS à cette fréquence)')
print(f'Fs correct          : {fs_correct} Hz → ✅ OK (> {2*f_signal} Hz)')
print(f'Fs trop bas         : {fs_trop_bas} Hz → ❌ VIOLATION (< {2*f_signal} Hz)')
print(f'\n👻 Fréquence fantôme (alias) : {f_alias} Hz')
print(f'   Le signal à {f_signal} Hz est perçu comme s\'il était à {f_alias} Hz !')

In [ ]:
# ===== DÉFI 3 : Visualisation de l'aliasing =====

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('DÉFI 3 — Démonstration de l\'Aliasing et du Théorème de Shannon', fontsize=13, fontweight='bold')

# Signal temporel - bon échantillonnage
axes[0,0].plot(t_continu, signal_continu, 'b-', alpha=0.3, label='Signal continu réel')
axes[0,0].stem(t_correct, signal_correct, linefmt='g-', markerfmt='go', basefmt='k-', label=f'Fs={fs_correct}Hz ✅')
axes[0,0].set_title(f'Bon échantillonnage (Fs={fs_correct} Hz > 2×{f_signal}={2*f_signal} Hz)')
axes[0,0].set_xlabel('Temps (s)'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# Signal temporel - mauvais échantillonnage
t_alias_continu = np.linspace(0, duree3, 10000)
signal_fantome = np.sin(2 * np.pi * f_alias * t_alias_continu)

axes[0,1].plot(t_continu, signal_continu, 'b-', alpha=0.3, label='Signal réel (300 Hz)')
axes[0,1].plot(t_alias_continu, signal_fantome, 'r-', alpha=0.5, label=f'Fréquence fantôme ({f_alias} Hz)')
axes[0,1].stem(t_alias, signal_alias, linefmt='r-', markerfmt='ro', basefmt='k-', label=f'Fs={fs_trop_bas}Hz ❌')
axes[0,1].set_title(f'Mauvais échantillonnage (Fs={fs_trop_bas} Hz < {2*f_signal} Hz)\n→ Aliasing ! Signal fantôme à {f_alias} Hz')
axes[0,1].set_xlabel('Temps (s)'); axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

# Spectre FFT - bon échantillonnage
N3 = len(signal_correct)
freqs3 = fftfreq(N3, d=1/fs_correct)[:N3//2]
spec3 = np.abs(fft(signal_correct))[:N3//2]
axes[1,0].plot(freqs3, spec3, color='green')
axes[1,0].axvline(f_signal, color='blue', linestyle='--', label=f'Vrai signal : {f_signal} Hz')
axes[1,0].set_title('Spectre FFT — Bon échantillonnage\n(un seul pic au bon endroit ✅)')
axes[1,0].set_xlabel('Fréquence (Hz)'); axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

# Spectre FFT - mauvais échantillonnage
Na = len(signal_alias)
freqsa = fftfreq(Na, d=1/fs_trop_bas)[:Na//2]
speca = np.abs(fft(signal_alias))[:Na//2]
axes[1,1].plot(freqsa, speca, color='red')
axes[1,1].axvline(f_alias, color='orange', linestyle='--', linewidth=2, label=f'Fréquence fantôme : {f_alias} Hz')
axes[1,1].set_title(f'Spectre FFT — Mauvais échantillonnage\n(pic à {f_alias} Hz au lieu de {f_signal} Hz ❌)')
axes[1,1].set_xlabel('Fréquence (Hz)'); axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/defi3_aliasing.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ===== DÉFI 3 : Correction par filtre anti-repliement + interpolation (sinc) =====

print('🔧 Correction de l\'aliasing par interpolation sinc...')

def interpolation_sinc(signal_souechantillonne, fs_bas, fs_haut):
    """
    Reconstruction du signal à fs_haut depuis fs_bas par convolution sinc.
    Le filtre sinc est le filtre idéal dans le domaine fréquentiel.
    """
    # Ratio de suréchantillonnage
    ratio = int(fs_haut / fs_bas)
    
    # Suréchantillonnage simple (insertion de zéros)
    signal_suretech = np.zeros(len(signal_souechantillonne) * ratio)
    signal_suretech[::ratio] = signal_souechantillonne
    
    # Filtre sinc = idéal pour reconstruction
    longueur_sinc = 101
    n = np.arange(-longueur_sinc//2, longueur_sinc//2 + 1)
    fc_norm = 1 / ratio  # Fréquence de coupure normalisée
    sinc_filtre = np.sinc(2 * fc_norm * n) * np.hanning(len(n))
    sinc_filtre /= np.sum(sinc_filtre)  # Normalisation
    
    # Convolution avec le filtre sinc
    signal_reconstruit = np.convolve(signal_suretech, sinc_filtre * ratio, mode='same')
    
    return signal_reconstruit

# Reconstruction du signal à partir de l'échantillonnage trop bas
ratio_echant = 16  # fs_haut = 500 * 16 = 8000 Hz
signal_reconstruit = interpolation_sinc(signal_alias, fs_trop_bas, fs_trop_bas * ratio_echant)
t_reconstruit = np.linspace(0, duree3, len(signal_reconstruit))

# Comparaison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('DÉFI 3 — Reconstruction par filtre sinc (interpolation idéale)', fontsize=13, fontweight='bold')

axes[0].plot(t_continu, signal_continu, 'b-', linewidth=2, label='Signal réel (300 Hz)', alpha=0.8)
axes[0].stem(t_alias, signal_alias, linefmt='r:', markerfmt='ro', basefmt='k-', label='Échantillons aliasés')
axes[0].set_title('Avant correction (échantillonnage aliasé)')
axes[0].set_xlabel('Temps (s)'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(t_continu, signal_continu, 'b-', linewidth=2, label='Signal réel', alpha=0.8)
axes[1].plot(t_reconstruit[:len(t_continu)], signal_reconstruit[:len(t_continu)], 'g--', 
            linewidth=1.5, label='Signal reconstruit (sinc)')
axes[1].set_title('Après reconstruction par interpolation sinc ✅')
axes[1].set_xlabel('Temps (s)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/defi3_reconstruction_sinc.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Défi 3 terminé — Les fréquences fantômes ont été éliminées !')

---
# 🔵 DÉFI 4 — Le Diagnostic du Futur (Smart Health / ECG)

**Contexte :** Signal ECG bruité (bruit musculaire haute fréquence + dérive basse fréquence respiratoire).  
**Mission :** Nettoyer le signal pour détecter les battements cardiaques (complexe QRS).

> 💡 **ECG = Électrocardiogramme :** Mesure de l'activité électrique du cœur.
> Le complexe QRS est le grand pic qui correspond à chaque battement.

In [ ]:
# ===== DÉFI 4 : Étape 1 — Génération d'un signal ECG synthétique =====

print('❤️ DÉFI 4 — Le Diagnostic du Futur (ECG)')
print('-' * 50)

fs4 = 360      # Fréquence d'échantillonnage ECG standard (360 Hz)
duree4 = 10    # 10 secondes
t4 = np.linspace(0, duree4, fs4 * duree4)

def ecg_synthetique(t, fs, bpm=75):
    """
    Génère un signal ECG simulé avec les ondes P, QRS et T.
    BPM = battements par minute (normal = 60-100)
    """
    periode = 60 / bpm  # Période en secondes entre deux battements
    ecg = np.zeros(len(t))
    
    for beat_start in np.arange(0, t[-1], periode):
        # Onde P (dépolarisation auriculaire) — petite bosse avant QRS
        idx_p = int((beat_start + 0.1) * fs)
        if idx_p < len(t):
            for i, v in enumerate([0.1, 0.2, 0.15, 0.1]):
                if idx_p + i < len(ecg):
                    ecg[idx_p + i] += v
        
        # Complexe QRS (dépolarisation ventriculaire) — grand pic caractéristique
        idx_qrs = int((beat_start + 0.25) * fs)
        if idx_qrs < len(t):
            for i, v in enumerate([-0.2, -0.3, 1.5, -0.5, -0.2, 0.1]):
                if idx_qrs + i < len(ecg):
                    ecg[idx_qrs + i] += v
        
        # Onde T (repolarisation ventriculaire) — bosse après QRS
        idx_t = int((beat_start + 0.45) * fs)
        if idx_t < len(t):
            for i, v in enumerate([0.1, 0.2, 0.3, 0.25, 0.15, 0.05]):
                if idx_t + i < len(ecg):
                    ecg[idx_t + i] += v
    
    # Lissage pour un signal plus réaliste
    from scipy.ndimage import gaussian_filter1d
    ecg = gaussian_filter1d(ecg, sigma=2)
    return ecg

# Signal ECG propre
ecg_propre = ecg_synthetique(t4, fs4, bpm=75)

# Ajout des bruits
np.random.seed(42)
bruit_emg = 0.15 * np.random.randn(len(t4))                     # Bruit musculaire (haute fréquence)
derive_respiratoire = 0.3 * np.sin(2 * np.pi * 0.2 * t4)       # Dérive respiratoire (0.2 Hz)

# Signal bruité complet
ecg_bruite = ecg_propre + bruit_emg + derive_respiratoire

print(f'✅ Signal ECG créé : {bpm := 75} BPM (rythme normal)')
print(f'   Bruit EMG (musculaire) : amplitude 0.15')
print(f'   Dérive respiratoire : 0.2 Hz, amplitude 0.3')

In [ ]:
# ===== DÉFI 4 : Étape 2 — Filtrage (Passe-Bas + Passe-Haut) =====

print('🔧 Application des filtres...')

# --- Filtre Passe-Bas (pour éliminer le bruit EMG haute fréquence) ---
# Fréquence de coupure = 40 Hz (le QRS est principalement entre 5-40 Hz)
fc_bas = 40  # Hz
ordre = 4    # Ordre du filtre (plus élevé = coupure plus nette)

b_pb, a_pb = signal.butter(ordre, fc_bas / (fs4/2), btype='low')
ecg_apres_pb = signal.filtfilt(b_pb, a_pb, ecg_bruite)

print(f'   ✅ Filtre Passe-Bas appliqué (fc = {fc_bas} Hz) → Bruit EMG réduit')

# --- Filtre Passe-Haut (pour éliminer la dérive respiratoire basse fréquence) ---
# Fréquence de coupure = 0.5 Hz (la respiration est à 0.2 Hz)
fc_haut = 0.5  # Hz

b_ph, a_ph = signal.butter(ordre, fc_haut / (fs4/2), btype='high')
ecg_filtre = signal.filtfilt(b_ph, a_ph, ecg_apres_pb)

print(f'   ✅ Filtre Passe-Haut appliqué (fc = {fc_haut} Hz) → Dérive respiratoire éliminée')

In [ ]:
# ===== DÉFI 4 : Étape 3 — Détection QRS et visualisation =====

# Détection des pics QRS (battements cardiaques)
seuil = 0.5 * np.max(ecg_filtre)  # Seuil adaptatif
pics_qrs, _ = signal.find_peaks(ecg_filtre, height=seuil, distance=int(0.5 * fs4))

bpm_detecte = len(pics_qrs) / (duree4 / 60)
print(f'🫀 Battements détectés : {len(pics_qrs)} en {duree4}s → {bpm_detecte:.1f} BPM')
print(f'   (Rythme normal : 60-100 BPM) → {"✅ Normal" if 60 <= bpm_detecte <= 100 else "⚠️ Anomalie"}')

# Visualisation complète
fig, axes = plt.subplots(4, 1, figsize=(16, 14))
fig.suptitle('DÉFI 4 — Filtrage du Signal ECG et Détection QRS', fontsize=14, fontweight='bold')

axes[0].plot(t4, ecg_bruite, color='red', linewidth=0.7)
axes[0].set_title('ECG Bruité (EMG + Dérive respiratoire)')
axes[0].set_ylabel('Amplitude'); axes[0].grid(True, alpha=0.3)

axes[1].plot(t4, ecg_apres_pb, color='orange', linewidth=0.7)
axes[1].set_title('Après Filtre Passe-Bas (40 Hz) — Bruit EMG éliminé')
axes[1].set_ylabel('Amplitude'); axes[1].grid(True, alpha=0.3)

axes[2].plot(t4, ecg_filtre, color='green', linewidth=0.8)
axes[2].set_title('Après Filtre Passe-Haut (0.5 Hz) — Dérive respiratoire éliminée')
axes[2].set_ylabel('Amplitude'); axes[2].grid(True, alpha=0.3)

axes[3].plot(t4, ecg_filtre, color='blue', linewidth=0.8, label='ECG filtré')
axes[3].plot(t4, ecg_propre, color='lightblue', linewidth=0.5, alpha=0.7, label='ECG original')
axes[3].plot(t4[pics_qrs], ecg_filtre[pics_qrs], 'r^', markersize=10, label=f'Pics QRS détectés ({len(pics_qrs)})')
axes[3].set_title(f'Détection des Battements Cardiaques → {bpm_detecte:.0f} BPM ❤️')
axes[3].set_xlabel('Temps (s)'); axes[3].set_ylabel('Amplitude')
axes[3].legend(); axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/defi4_ecg.png', dpi=150, bbox_inches='tight')
plt.show()

# Analyse spectrale pour justifier les fréquences de coupure
freqs4 = fftfreq(len(ecg_bruite), d=1/fs4)[:len(ecg_bruite)//2]
spec_bruite = np.abs(fft(ecg_bruite))[:len(ecg_bruite)//2]
spec_filtre = np.abs(fft(ecg_filtre))[:len(ecg_filtre)//2]

plt.figure(figsize=(12, 4))
plt.plot(freqs4[freqs4 < 100], spec_bruite[freqs4 < 100], color='red', alpha=0.7, label='ECG bruité')
plt.plot(freqs4[freqs4 < 100], spec_filtre[freqs4 < 100], color='green', label='ECG filtré')
plt.axvline(0.5, color='orange', linestyle='--', label='Coupure haute-passe 0.5 Hz')
plt.axvline(40, color='purple', linestyle='--', label='Coupure basse-passe 40 Hz')
plt.title('Spectre FFT — Justification des fréquences de coupure')
plt.xlabel('Fréquence (Hz)'); plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig('/content/defi4_spectre_ecg.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Défi 4 terminé — Le signal ECG est propre et les battements sont détectés !')

---
# 🟣 DÉFI 5 — L'Opticien Numérique (Déconvolution / Flou d'image)

**Contexte :** Une radiographie est floue à cause d'un mouvement du patient.  
**Mission :** Restaurer la netteté via le Filtre de Wiener.

> 💡 **Le flou = convolution spatiale :** En Fourier, la convolution devient une multiplication.
> Déconvolution = diviser dans le domaine de Fourier pour retrouver l'image originale.

In [ ]:
# ===== DÉFI 5 : Déconvolution d'image =====
from skimage import data, color
from scipy.signal import convolve2d

print('🔬 DÉFI 5 — L\'Opticien Numérique (Déconvolution)')
print('-' * 55)

# Charger une image de test
image_originale = color.rgb2gray(data.astronaut()).astype(np.float64)
image_originale = image_originale[:200, :200]  # Garder une portion pour aller plus vite
image_originale /= image_originale.max()       # Normaliser 0-1

print(f'✅ Image chargée : {image_originale.shape[0]}×{image_originale.shape[1]} pixels')

# ===== Étape 1 : Modéliser le flou (PSF = Point Spread Function) =====
# Le flou de bougé directionnel = une ligne horizontale (mouvement vers la droite)

def creer_psf_mouvement(taille, angle_deg=0, longueur=15):
    """
    Crée une PSF (noyau de flou) simulant un mouvement.
    taille : taille du noyau (carré)
    angle_deg : direction du mouvement en degrés
    longueur : distance du mouvement en pixels
    """
    psf = np.zeros((taille, taille))
    centre = taille // 2
    angle_rad = np.deg2rad(angle_deg)
    
    for i in range(longueur):
        x = centre + int(i * np.cos(angle_rad))
        y = centre + int(i * np.sin(angle_rad))
        if 0 <= x < taille and 0 <= y < taille:
            psf[y, x] = 1
    
    psf /= psf.sum()  # Normaliser
    return psf

# PSF = flou de mouvement horizontal de 15 pixels
taille_psf = 31
psf = creer_psf_mouvement(taille_psf, angle_deg=0, longueur=15)

# ===== Étape 2 : Appliquer le flou (convolution) =====
image_floue = convolve2d(image_originale, psf, mode='same', boundary='wrap')

# Ajouter un petit bruit pour simuler la réalité
np.random.seed(42)
image_floue += 0.01 * np.random.randn(*image_floue.shape)

print(f'✅ Flou de mouvement appliqué (horizontal, 15 pixels)')
print(f'   Rappel : flou = convolution spatiale = multiplication dans Fourier')

In [ ]:
# ===== DÉFI 5 : Filtre de Wiener pour déconvolution =====

print('🔧 Application du Filtre de Wiener...')

def filtre_wiener_2d(image_floue, psf, K=0.01):
    """
    Filtre de Wiener pour déconvolution :
    - K = paramètre de régularisation (évite l'amplification du bruit)
    - Plus K est petit, plus on essaie de retrouver le signal original
    - Plus K est grand, plus on préserve la stabilité (mais moins net)
    
    Formule : H_wiener = H* / (|H|² + K)
    où H = FFT de la PSF
    """
    # FFT de l'image floue et de la PSF
    IMAGE_FLOUE_FFT = np.fft.fft2(image_floue)
    
    # La PSF doit avoir la même taille que l'image
    psf_padded = np.zeros_like(image_floue)
    cy, cx = image_floue.shape[0]//2, image_floue.shape[1]//2
    ph, pw = psf.shape[0]//2, psf.shape[1]//2
    psf_padded[cy-ph:cy+ph+1, cx-pw:cx+pw+1] = psf
    
    PSF_FFT = np.fft.fft2(psf_padded)
    
    # Filtre de Wiener : H* / (|H|² + K)
    PSF_CONJ = np.conj(PSF_FFT)                  # Conjugué complexe
    PSF_MOD2 = np.abs(PSF_FFT) ** 2              # Module au carré
    WIENER = PSF_CONJ / (PSF_MOD2 + K)           # Filtre de Wiener
    
    # Application du filtre
    IMAGE_RESTAUREE_FFT = IMAGE_FLOUE_FFT * WIENER
    
    # Retour dans le domaine spatial
    image_restauree = np.real(np.fft.ifft2(IMAGE_RESTAUREE_FFT))
    
    # Clipping entre 0 et 1
    image_restauree = np.clip(image_restauree, 0, 1)
    
    return image_restauree

# Appliquer le filtre de Wiener
image_restauree = filtre_wiener_2d(image_floue, psf, K=0.01)
print('✅ Déconvolution terminée !')

# ===== Visualisation =====
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('DÉFI 5 — Déconvolution d\'image (Filtre de Wiener)', fontsize=14, fontweight='bold')

axes[0].imshow(image_originale, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Image Originale (référence)')
axes[0].axis('off')

axes[1].imshow(image_floue, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Image Floue (flou de bougé 15px)\n→ Comme une radio mal prise')
axes[1].axis('off')

axes[2].imshow(image_restauree, cmap='gray', vmin=0, vmax=1)
axes[2].set_title('Image Restaurée (Filtre de Wiener K=0.01) ✅\n→ Contours retrouvés')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('/content/defi5_deconvolution.png', dpi=150, bbox_inches='tight')
plt.show()

# Mesure de qualité : PSNR (Peak Signal-to-Noise Ratio)
def psnr(img1, img2):
    mse = np.mean((img1 - img2)**2)
    if mse == 0: return float('inf')
    return 10 * np.log10(1.0 / mse)

psnr_flou = psnr(image_originale, image_floue)
psnr_restaure = psnr(image_originale, image_restauree)
print(f'📊 Qualité de restauration :')
print(f'   PSNR image floue      : {psnr_flou:.2f} dB')
print(f'   PSNR image restaurée  : {psnr_restaure:.2f} dB')
print(f'   Amélioration          : +{psnr_restaure - psnr_flou:.2f} dB')
print('✅ Défi 5 terminé !')

---
# 🩷 DÉFI 6 — Le Chasseur de Fantômes (Aliasing & Bruit ISO)

**Contexte :** Une photo de nuit est gâchée par le bruit ISO (grain électronique) et des pixels aberrants.  
**Mission :** Nettoyer l'image avec un filtre hybride (fréquentiel + spatial).

In [ ]:
# ===== DÉFI 6 : Réduction du bruit d'image =====
from skimage import data, color, img_as_float
from skimage.filters import gaussian
from scipy.ndimage import median_filter

print('👻 DÉFI 6 — Le Chasseur de Fantômes (Bruit ISO)')
print('-' * 52)

# Charger une image de test
image6 = img_as_float(data.coins())
print(f'✅ Image chargée : {image6.shape} pixels')

# Simuler du bruit ISO (bruit gaussien + pixels aberrants)
np.random.seed(42)

# Bruit gaussien (grain numérique)
bruit_gaussien = 0.1 * np.random.randn(*image6.shape)

# Pixels aberrants ("sel et poivre" = pixels aléatoires très clairs ou très sombres)
pixels_aberrants = np.zeros_like(image6)
nb_pixels_sel = int(0.02 * image6.size)  # 2% de pixels aberrants
coords_y = np.random.randint(0, image6.shape[0], nb_pixels_sel)
coords_x = np.random.randint(0, image6.shape[1], nb_pixels_sel)
pixels_aberrants[coords_y[:nb_pixels_sel//2], coords_x[:nb_pixels_sel//2]] = 1.0   # Blanc
pixels_aberrants[coords_y[nb_pixels_sel//2:], coords_x[nb_pixels_sel//2:]] = -1.0  # Noir

image6_bruitee = np.clip(image6 + bruit_gaussien + pixels_aberrants, 0, 1)
print(f'   Bruit gaussien ajouté (amplitude 0.1)')
print(f'   Pixels aberrants : {nb_pixels_sel} pixels (2%)')

In [ ]:
# ===== DÉFI 6 : Analyse spectrale + Filtrage hybride =====

# --- Étape 1 : Analyse spectrale (FFT 2D) ---
print('🔬 Analyse spectrale 2D (FFT 2D)...')
FFT2D_bruitee = np.fft.fftshift(np.fft.fft2(image6_bruitee))  # FFT 2D centrée
spectre_2d = np.log(np.abs(FFT2D_bruitee) + 1)                 # Échelle log pour mieux voir

# --- Étape 2 : Filtre Passe-Bas fréquentiel (masque circulaire) ---
print('🔧 Application du filtre Passe-Bas fréquentiel...')

def masque_passe_bas(taille_h, taille_w, rayon):
    """
    Crée un masque circulaire dans le domaine de Fourier.
    Les fréquences à l'intérieur du cercle (basses fréquences) passent.
    Les fréquences à l'extérieur (hautes fréquences = bruit) sont bloquées.
    """
    cy, cx = taille_h // 2, taille_w // 2
    y, x = np.ogrid[:taille_h, :taille_w]
    distance = np.sqrt((y - cy)**2 + (x - cx)**2)
    masque = (distance <= rayon).astype(float)
    return masque

rayon_filtre = 60  # Garder les fréquences jusqu'à rayon=60 (empirique)
masque = masque_passe_bas(*image6_bruitee.shape, rayon_filtre)
FFT2D_filtre = FFT2D_bruitee * masque
image6_filtre_freq = np.real(np.fft.ifft2(np.fft.ifftshift(FFT2D_filtre)))
image6_filtre_freq = np.clip(image6_filtre_freq, 0, 1)

print(f'   ✅ Masque circulaire passe-bas (rayon={rayon_filtre} pixels dans l\'espace de Fourier)')

# --- Étape 3 : Filtre Médian spatial (élimine les pixels aberrants) ---
print('🔧 Application du filtre Médian spatial...')
image6_mediane = median_filter(image6_bruitee, size=3)
print('   ✅ Filtre médian (taille 3×3) — Pixels aberrants éliminés')

# --- Étape 4 : Combinaison hybride ---
image6_hybride = np.clip(0.5 * image6_filtre_freq + 0.5 * image6_mediane, 0, 1)

In [ ]:
# ===== DÉFI 6 : Visualisation =====

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('DÉFI 6 — Réduction du bruit ISO (Filtrage Hybride)', fontsize=14, fontweight='bold')

axes[0,0].imshow(image6, cmap='gray'); axes[0,0].set_title('Image originale'); axes[0,0].axis('off')
axes[0,1].imshow(image6_bruitee, cmap='gray'); axes[0,1].set_title('Image bruitée (ISO simulé)'); axes[0,1].axis('off')
axes[0,2].imshow(spectre_2d, cmap='hot')
axes[0,2].set_title('Spectre 2D (FFT)\n→ Hautes fréquences = bruit au bord')
axes[0,2].axis('off')

axes[1,0].imshow(image6_filtre_freq, cmap='gray')
axes[1,0].set_title('Après filtre Passe-Bas fréquentiel\n(bruit gaussien réduit)')
axes[1,0].axis('off')

axes[1,1].imshow(image6_mediane, cmap='gray')
axes[1,1].set_title('Après filtre Médian spatial\n(pixels aberrants éliminés)')
axes[1,1].axis('off')

axes[1,2].imshow(image6_hybride, cmap='gray')
axes[1,2].set_title('Résultat hybride (PB + Médian) ✅\n→ Meilleur des deux mondes')
axes[1,2].axis('off')

plt.tight_layout()
plt.savefig('/content/defi6_bruit_iso.png', dpi=150, bbox_inches='tight')
plt.show()

# Métriques
psnr_bruite = psnr(image6, image6_bruitee)
psnr_hybride = psnr(image6, image6_hybride)
print(f'📊 PSNR image bruitée  : {psnr_bruite:.2f} dB')
print(f'   PSNR image hybride  : {psnr_hybride:.2f} dB  (+{psnr_hybride-psnr_bruite:.2f} dB)')
print('✅ Défi 6 terminé !')

---
# 🤍 DÉFI 7 — L'Espion Fréquentiel (Stéganographie)

**Contexte :** Cacher un message secret dans une image sans que ça se voie.  
**Mission :** Implémenter la stéganographie par manipulation des bits de poids faibles (LSB).

> 💡 **LSB (Least Significant Bit) :** Chaque pixel est codé sur 8 bits (0-255).
> Le dernier bit (bit 0) a peu d'impact visuel. On peut y cacher de l'information !

In [ ]:
# ===== DÉFI 7 : Stéganographie par LSB =====
from PIL import Image as PILImage

print('🕵️ DÉFI 7 — L\'Espion Fréquentiel (Stéganographie LSB)')
print('-' * 58)

# ===== Étape 1 : Image hôte =====
image_hote_np = (data.astronaut()[:200, :200]).copy()  # Image couleur RGB uint8
print(f'✅ Image hôte : {image_hote_np.shape}  (RGB, 0-255)')

# ===== Étape 2 : Message secret =====
message_secret = "SupNum TMA 2026 - Message Secret!"
print(f'✅ Message à cacher : "{message_secret}"')
print(f'   Longueur : {len(message_secret)} caractères')

# ===== Étape 3 : Encodage du message en bits =====
def texte_vers_bits(texte):
    """Convertit un texte en une liste de bits (0 et 1)."""
    bits = []
    # On ajoute d'abord la longueur du message (16 bits pour supporter jusqu'à 65535 chars)
    longueur = len(texte)
    for i in range(16):
        bits.append((longueur >> (15 - i)) & 1)
    # Puis les bits de chaque caractère
    for char in texte:
        code = ord(char)  # Code ASCII du caractère
        for i in range(8):
            bits.append((code >> (7 - i)) & 1)
    return bits

def bits_vers_texte(bits):
    """Convertit une liste de bits en texte."""
    # Lire la longueur (16 premiers bits)
    longueur = 0
    for i in range(16):
        longueur = (longueur << 1) | bits[i]
    # Lire le message
    texte = ''
    for i in range(longueur):
        code = 0
        for j in range(8):
            code = (code << 1) | bits[16 + i*8 + j]
        texte += chr(code)
    return texte

bits_message = texte_vers_bits(message_secret)
print(f'\n   Représentation binaire (premiers 32 bits) :')
print(f'   {bits_message[:32]}')
print(f'   Total : {len(bits_message)} bits à cacher')

In [ ]:
# ===== DÉFI 7 : Encodage LSB (cacher le message) =====

def cacher_message_lsb(image, bits):
    """
    Cache les bits du message dans le bit de poids faible (LSB) de chaque pixel.
    Canaux RGB dans l'ordre : R d'abord, puis G, puis B.
    """
    image_stego = image.copy().astype(np.uint8)
    h, w, c = image_stego.shape
    nb_bits = len(bits)
    
    assert nb_bits <= h * w * c, 'Image trop petite pour ce message !'
    
    idx = 0  # Index du bit courant
    for i in range(h):
        for j in range(w):
            for k in range(c):
                if idx >= nb_bits:
                    break
                pixel = int(image_stego[i, j, k])
                # Remplacer le LSB par le bit du message
                # & 0b11111110 : mettre le LSB à 0
                # | bits[idx]  : mettre le bit du message
                image_stego[i, j, k] = (pixel & 0b11111110) | bits[idx]
                idx += 1
    
    return image_stego

def extraire_message_lsb(image_stego):
    """Extrait les bits cachés dans les LSB d'une image stégo."""
    h, w, c = image_stego.shape
    bits = []
    
    for i in range(h):
        for j in range(w):
            for k in range(c):
                bits.append(int(image_stego[i, j, k]) & 1)  # Extraire le LSB
    
    return bits_vers_texte(bits)

print('🔒 Encodage du message secret...')
image_stego = cacher_message_lsb(image_hote_np, bits_message)
print('✅ Message caché dans l\'image !')

print('\n🔓 Décodage du message secret...')
message_extrait = extraire_message_lsb(image_stego)
print(f'   Message extrait : "{message_extrait}"')
print(f'   ✅ Correspondance : {message_extrait == message_secret}')

In [ ]:
# ===== DÉFI 7 : Visualisation - L'image ne doit pas changer à l'oeil =====

diff = np.abs(image_hote_np.astype(float) - image_stego.astype(float))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('DÉFI 7 — Stéganographie par LSB (le message est invisible !)', fontsize=13, fontweight='bold')

axes[0].imshow(image_hote_np)
axes[0].set_title('Image Hôte (originale)')
axes[0].axis('off')

axes[1].imshow(image_stego)
axes[1].set_title(f'Image Stégo (avec message caché)\n→ Identique à l\'œil humain !')
axes[1].axis('off')

# Différence amplifiée × 255 pour la rendre visible
diff_amplifiee = np.clip(diff * 255, 0, 255).astype(np.uint8)
axes[2].imshow(diff_amplifiee)
axes[2].set_title(f'Différence amplifiée ×255\n→ Max changement = {diff.max():.0f} (sur 255 !)')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('/content/defi7_steganographie.png', dpi=150, bbox_inches='tight')
plt.show()

# Statistiques
print(f'\n📊 Invisibilité de la stéganographie :')
print(f'   Changement max par pixel : {diff.max():.0f} sur 255')
print(f'   Changement moyen         : {diff.mean():.4f}')
print(f'   PSNR (mesure qualité)    : {psnr(image_hote_np/255, image_stego/255):.2f} dB')
print(f'\n✅ Message : "{message_extrait}" correctement caché et extrait !')
print('✅ Défi 7 terminé !')

---
# 🖤 DÉFI 8 — Le Compresseur Intelligent (Masquage Psychoacoustique)

**Contexte :** Compresser un signal audio en exploitant les limites de l'ouïe humaine.  
**Mission :** Simuler un moteur de compression type MP3 (STFT + masquage fréquentiel).

> 💡 **Masquage psychoacoustique :** Un son fort rend inaudibles les sons faibles voisins en fréquence.
> Le MP3 exploite cette limite pour supprimer les données que l'oreille n'entend pas de toute façon.

In [ ]:
# ===== DÉFI 8 : Compression par masquage psychoacoustique =====

print('🎵 DÉFI 8 — Le Compresseur Intelligent (Masquage Psychoacoustique)')
print('-' * 65)

# Signal test : mélange de plusieurs fréquences (simule de la musique)
fs8 = 44100   # CD quality
duree8 = 2    # 2 secondes
t8 = np.linspace(0, duree8, fs8 * duree8)

# Signal audio complexe ("musique" simulée)
# Un son fort à 1000 Hz doit masquer les sons faibles à 1100 Hz et 900 Hz
signal8 = (
    1.0  * np.sin(2 * np.pi * 440  * t8) +   # Note La (440 Hz) — forte
    1.5  * np.sin(2 * np.pi * 1000 * t8) +   # 1000 Hz — très forte (masquante)
    0.1  * np.sin(2 * np.pi * 1100 * t8) +   # 1100 Hz — faible (sera masquée)
    0.08 * np.sin(2 * np.pi * 900  * t8) +   # 900 Hz — très faible (sera masquée)
    0.5  * np.sin(2 * np.pi * 2000 * t8) +   # 2000 Hz — moyenne
    0.3  * np.sin(2 * np.pi * 3000 * t8)     # 3000 Hz — douce
)
signal8 /= np.max(np.abs(signal8))  # Normalisation

print('✅ Signal audio créé :')
print('   440 Hz (forte), 1000 Hz (très forte), 1100 Hz & 900 Hz (faibles = masquées)...')

In [ ]:
# ===== DÉFI 8 : STFT + Modèle de masquage fréquentiel =====

# --- Étape 1 : STFT (Short-Time Fourier Transform) ---
print('🔬 Calcul de la STFT...')

taille_fenetre = 2048   # Taille de chaque fenêtre d'analyse
chevauchement8 = 512    # Chevauchement entre fenêtres

# Calcul de la STFT
f_stft, t_stft, Zxx = signal.stft(signal8, fs8, nperseg=taille_fenetre, noverlap=chevauchement8)
magnitude = np.abs(Zxx)     # Amplitudes
phase = np.angle(Zxx)       # Phases

print(f'   ✅ STFT calculée : {magnitude.shape[0]} bins fréquence × {magnitude.shape[1]} trames temps')

# --- Étape 2 : Modèle de Masquage Fréquentiel ---
print('\n🔧 Application du masquage psychoacoustique...')

def masquage_frequentiel(magnitude, seuil_db=-20, largeur_masque=5):
    """
    Modèle simplifié de masquage fréquentiel :
    - Pour chaque fréquence, si son amplitude est fort, elle masque les voisines faibles
    - On supprime (met à 0) les fréquences sous le seuil de masquage
    
    seuil_db : en dessous de ce niveau (par rapport au maximum), la fréquence est masquée
    largeur_masque : nombre de bins de fréquence masqués autour du pic fort
    """
    magnitude_masquee = magnitude.copy()
    nb_trames = magnitude.shape[1]
    
    for trame in range(nb_trames):
        spectre_trame = magnitude[:, trame]
        max_amp = np.max(spectre_trame)
        
        if max_amp < 1e-10:
            continue
        
        # Seuil d'audibilité = max_amp × 10^(seuil_db/20)
        seuil_amplitude = max_amp * 10 ** (seuil_db / 20)
        
        # Supprimer les fréquences sous le seuil (inaudibles)
        masque_inaudible = spectre_trame < seuil_amplitude
        magnitude_masquee[masque_inaudible, trame] = 0
    
    return magnitude_masquee

magnitude_comprimee = masquage_frequentiel(magnitude, seuil_db=-25)

# Calculer le taux de compression
nb_total = magnitude.size
nb_non_zero = np.count_nonzero(magnitude_comprimee)
taux_compression = (1 - nb_non_zero / nb_total) * 100

print(f'   ✅ Masquage appliqué')
print(f'   Coefficients supprimés : {taux_compression:.1f}%')
print(f'   → Réduction de taille approximative : {taux_compression:.1f}%')

In [ ]:
# ===== DÉFI 8 : Reconstruction + Visualisation =====

# Reconstruction du signal comprimé via STFT inverse
Zxx_comprime = magnitude_comprimee * np.exp(1j * phase)
_, signal8_reconstruit = signal.istft(Zxx_comprime, fs8, nperseg=taille_fenetre, noverlap=chevauchement8)

# Aligner les longueurs
longueur_commune = min(len(signal8), len(signal8_reconstruit))
signal8 = signal8[:longueur_commune]
signal8_reconstruit = signal8_reconstruit[:longueur_commune]

# Visualisation
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('DÉFI 8 — Compression par Masquage Psychoacoustique (MP3 Light)', fontsize=13, fontweight='bold')

# Spectrogramme original
axes[0,0].pcolormesh(t_stft, f_stft[:100], 20*np.log10(magnitude[:100]+1e-10), shading='auto', cmap='viridis')
axes[0,0].set_title('Spectrogramme STFT (original)')
axes[0,0].set_xlabel('Temps (s)'); axes[0,0].set_ylabel('Fréquence (Hz)')

# Spectrogramme comprimé
axes[0,1].pcolormesh(t_stft, f_stft[:100], 20*np.log10(magnitude_comprimee[:100]+1e-10), shading='auto', cmap='viridis')
axes[0,1].set_title(f'Spectrogramme comprimé ({taux_compression:.1f}% supprimé)')
axes[0,1].set_xlabel('Temps (s)'); axes[0,1].set_ylabel('Fréquence (Hz)')

# Signaux temporels
nb8 = int(0.01 * fs8)
axes[1,0].plot(t8[:nb8], signal8[:nb8], color='blue', linewidth=0.8)
axes[1,0].set_title('Signal original')
axes[1,0].set_xlabel('Temps (s)'); axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(t8[:nb8], signal8_reconstruit[:nb8], color='green', linewidth=0.8)
axes[1,1].set_title(f'Signal reconstruit après compression ✅')
axes[1,1].set_xlabel('Temps (s)'); axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/defi8_compression.png', dpi=150, bbox_inches='tight')
plt.show()

# Illustration du masquage fréquentiel
plt.figure(figsize=(14, 5))
freqs8 = fftfreq(len(signal8), d=1/fs8)[:len(signal8)//2]
spec8_orig = np.abs(fft(signal8))[:len(signal8)//2]
spec8_comp = np.abs(fft(signal8_reconstruit))[:len(signal8_reconstruit)//2]

plt.plot(freqs8[freqs8 < 5000], spec8_orig[freqs8 < 5000], color='blue', label='Original', alpha=0.8)
plt.plot(freqs8[freqs8 < 5000], spec8_comp[freqs8 < 5000], color='green', linewidth=1.5, label='Comprimé')
plt.axvline(1000, color='red', linestyle='--', alpha=0.5, label='1000 Hz (fort = masquant)')
plt.axvline(1100, color='orange', linestyle='--', alpha=0.5, label='1100 Hz (faible = masqué)')
plt.title('Illustration du Masquage Fréquentiel\n→ Les composantes faibles autour du 1000 Hz sont supprimées')
plt.xlabel('Fréquence (Hz)'); plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig('/content/defi8_masquage.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📊 Résumé compression :')
print(f'   Taux de réduction des données : {taux_compression:.1f}%')
print(f'   Corrélation original/reconstruit : {np.corrcoef(signal8, signal8_reconstruit)[0,1]:.4f}')
print('✅ Défi 8 terminé !')

---
# 🟫 DÉFI 9 — L'Annulateur d'Écho (Acoustic Echo Cancellation)

**Contexte :** Le son du haut-parleur est capté par le micro et crée un écho en boucle.  
**Mission :** Modéliser l'écho et le soustraire du signal pour isoler la voix.

> 💡 **La réponse impulsionnelle** d'une pièce décrit comment les sons se réverbèrent.
> L'écho = convolution du signal de référence avec cette réponse impulsionnelle.

In [ ]:
# ===== DÉFI 9 : Annulation d'écho acoustique =====

print('🔊 DÉFI 9 — L\'Annulateur d\'Écho (AEC)')
print('-' * 48)

fs9 = 8000   # Téléphonie standard
duree9 = 3   # 3 secondes
t9 = np.linspace(0, duree9, fs9 * duree9)

# --- Voix de l'étudiant (signal utile à préserver) ---
voix_etudiant = (
    np.sin(2 * np.pi * 180 * t9) * np.exp(-0.5 * t9) +
    0.5 * np.sin(2 * np.pi * 360 * t9) * np.exp(-0.5 * t9) +
    0.2 * np.random.randn(len(t9))
)
voix_etudiant /= np.max(np.abs(voix_etudiant))

# --- Signal de référence = ce qui sort du haut-parleur ---
# (voix du professeur, musique, etc.)
reference = (
    np.sin(2 * np.pi * 300 * t9) +
    0.5 * np.sin(2 * np.pi * 600 * t9)
)
reference /= np.max(np.abs(reference))

# --- Réponse Impulsionnelle (RI) de la pièce ---
# Modèle simplifié : écho direct + quelques réflexions
longueur_ri = int(0.1 * fs9)  # 100 ms de réverbération
ri_piece = np.zeros(longueur_ri)
ri_piece[0] = 0.7                               # Réflexion directe (70%)
ri_piece[int(0.02 * fs9)] = 0.3                 # 1ère réflexion (mur proche)
ri_piece[int(0.05 * fs9)] = 0.15               # 2ème réflexion
ri_piece[int(0.08 * fs9)] = 0.05               # 3ème réflexion (atténuée)

# --- Signal capté par le micro = voix + écho du haut-parleur ---
echo = np.convolve(reference, ri_piece, mode='full')[:len(t9)]
signal_micro = voix_etudiant + 0.8 * echo  # 80% de l'écho se retrouve dans le micro

print('✅ Simulation de la situation d\'écho :')
print(f'   Voix étudiant : signal utile à préserver')
print(f'   Haut-parleur  : signal de référence connu')
print(f'   Réponse impulsionnelle : {longueur_ri} échantillons ({longueur_ri/fs9*1000:.0f} ms)')
print(f'   Micro capte   : voix + écho (indésirable)')

In [ ]:
# ===== DÉFI 9 : Annulation de l'écho =====

print('\n🔧 Annulation de l\'écho...')

# Méthode 1 : Si on connaît la RI exacte
# On recalcule l'écho estimé et on le soustrait
echo_estime = np.convolve(reference, ri_piece, mode='full')[:len(t9)]
signal_sans_echo_exact = signal_micro - 0.8 * echo_estime

# Méthode 2 : Annulation par corrélation croisée (estimation de la RI)
# En pratique, on ne connaît pas la RI → on l'estime
print('   Estimation de la Réponse Impulsionnelle par corrélation croisée...')

def estimer_ri_lms(signal_micro, reference, longueur_filtre=400, mu=0.01):
    """
    Algorithme LMS (Least Mean Squares) pour estimer la RI :
    - Filtre adaptatif qui apprend la RI au fil du temps
    - mu = pas d'apprentissage (trop grand = instable, trop petit = lent)
    """
    N = len(signal_micro)
    w = np.zeros(longueur_filtre)   # Coefficients du filtre (RI estimée)
    signal_annule = np.zeros(N)     # Signal après annulation
    erreurs = np.zeros(N)
    
    for n in range(longueur_filtre, N):
        # Fenêtre du signal de référence
        x = reference[n - longueur_filtre:n][::-1]
        
        # Estimation de l'écho avec le filtre actuel
        echo_estime_n = np.dot(w, x)
        
        # Erreur = signal micro - écho estimé
        erreur = signal_micro[n] - echo_estime_n
        erreurs[n] = erreur
        
        # Mise à jour des coefficients (règle LMS)
        w = w + mu * erreur * x
        signal_annule[n] = erreur
    
    return signal_annule, w

signal_sans_echo_lms, ri_estimee = estimer_ri_lms(signal_micro, reference, longueur_filtre=200, mu=0.005)
print('   ✅ Algorithme LMS terminé — RI apprise de manière adaptive')
print('   ✅ Écho annulé !')

In [ ]:
# ===== DÉFI 9 : Visualisation =====

fig, axes = plt.subplots(3, 2, figsize=(16, 12))
fig.suptitle('DÉFI 9 — Annulation d\'Écho Acoustique (AEC)', fontsize=14, fontweight='bold')

nb9 = int(0.5 * fs9)

axes[0,0].plot(t9[:nb9], voix_etudiant[:nb9], color='blue')
axes[0,0].set_title('Voix de l\'étudiant (signal utile)')
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(t9[:nb9], reference[:nb9], color='orange')
axes[0,1].set_title('Signal de référence (haut-parleur)')
axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(t9[:nb9], signal_micro[:nb9], color='red')
axes[1,0].set_title('Signal capté par le micro\n(voix + écho — bruyant !)')
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(np.arange(len(ri_piece))/fs9*1000, ri_piece, color='purple', label='RI réelle')
axes[1,1].plot(np.arange(len(ri_estimee[:len(ri_piece)]))/fs9*1000,
              ri_estimee[:len(ri_piece)] * np.max(ri_piece)/max(np.max(np.abs(ri_estimee[:len(ri_piece)])),1e-10),
              color='green', linestyle='--', label='RI estimée (LMS)')
axes[1,1].set_title('Réponse Impulsionnelle de la pièce\n(réelle vs estimée par LMS)')
axes[1,1].set_xlabel('Temps (ms)'); axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

axes[2,0].plot(t9[:nb9], voix_etudiant[:nb9], color='blue', alpha=0.5, label='Voix originale')
axes[2,0].plot(t9[:nb9], signal_sans_echo_exact[:nb9], color='green', label='Après annulation exacte')
axes[2,0].set_title('Annulation Exacte (RI connue) ✅')
axes[2,0].legend(); axes[2,0].grid(True, alpha=0.3)

axes[2,1].plot(t9[:nb9], voix_etudiant[:nb9], color='blue', alpha=0.5, label='Voix originale')
axes[2,1].plot(t9[:nb9], signal_sans_echo_lms[:nb9], color='darkgreen', label='Après LMS adaptatif')
axes[2,1].set_title('Annulation Adaptative LMS (RI estimée) ✅')
axes[2,1].legend(); axes[2,1].grid(True, alpha=0.3)

for ax in axes.flat:
    ax.set_xlabel('Temps (s)')
    ax.set_ylabel('Amplitude')

plt.tight_layout()
plt.savefig('/content/defi9_annulation_echo.png', dpi=150, bbox_inches='tight')
plt.show()

# SNR
snr_avant = calculer_snr(voix_etudiant, signal_micro)
snr_apres = calculer_snr(voix_etudiant[200:], signal_sans_echo_lms[200:])
print(f'\n📊 SNR avant annulation : {snr_avant:.2f} dB')
print(f'   SNR après annulation  : {snr_apres:.2f} dB')
print('✅ Défi 9 terminé !')

---
# 🔶 DÉFI 10 — Le Radar Numérique (Corrélation et Détection)

**Contexte :** Un signal de référence est noyé dans du bruit. On doit détecter sa présence et sa position.  
**Mission :** Utiliser la corrélation croisée comme un "Matched Filter" (filtre adapté).

> 💡 **Corrélation croisée :** Mesure la similarité entre deux signaux à différents décalages temporels.
> Un pic de corrélation indique que le motif a été trouvé à cet instant précis.

In [ ]:
# ===== DÉFI 10 : Radar Numérique — Corrélation et Détection =====

print('📡 DÉFI 10 — Le Radar Numérique')
print('-' * 40)

# --- Étape 1 : Créer le motif de référence (code radar) ---
# On utilise un signal chirp (fréquence croissante) — typique des radars
fs10 = 10000  # Hz

duree_motif = 0.01   # Motif de 10 ms
t_motif = np.linspace(0, duree_motif, int(fs10 * duree_motif))
motif = signal.chirp(t_motif, f0=500, f1=2000, t1=duree_motif, method='linear')
motif /= np.max(np.abs(motif))

print(f'✅ Motif de référence : signal chirp de {duree_motif*1000:.0f}ms ({500} → {2000} Hz)')

# --- Étape 2 : Créer le signal reçu (motif + bruit fort) ---
duree10 = 0.5   # Signal reçu de 500 ms
N10 = int(fs10 * duree10)
t10 = np.linspace(0, duree10, N10)

# Bruit fort (simule l'environnement hostile)
snr_cible_db = -10  # SNR = -10 dB (bruit 10× plus fort que le signal !)
np.random.seed(42)
bruit10 = np.random.randn(N10)

# Injecter le motif à un endroit précis (inconnu a priori)
position_vraie = int(0.23 * N10)  # Position inconnue à détecter
signal10 = bruit10.copy()
puissance_bruit = np.var(bruit10)
amplitude_signal = np.sqrt(puissance_bruit * 10**(snr_cible_db/10))
signal10[position_vraie:position_vraie + len(motif)] += amplitude_signal * motif

print(f'\n   Signal reçu : {duree10*1000:.0f}ms de bruit avec le motif caché à t={position_vraie/fs10*1000:.1f}ms')
print(f'   SNR = {snr_cible_db} dB (le signal est {10**(abs(snr_cible_db)/10):.0f}× plus faible que le bruit !)')

In [ ]:
# ===== DÉFI 10 : Corrélation croisée par FFT =====

print('🔧 Calcul de la Corrélation Croisée via FFT (Matched Filter)...')

def correlation_croisee_fft(signal_recu, motif_reference):
    """
    Corrélation croisée rapide via FFT :
    - Corrélation(x, y) = IFFT(FFT(x) × CONJ(FFT(y)))
    - Mathématiquement : corrélation = convolution avec y retourné
    - Via FFT : beaucoup plus rapide que le calcul direct !
    """
    N = len(signal_recu) + len(motif_reference) - 1  # Longueur de la corrélation
    
    # Calcul FFT (avec zero-padding pour éviter les effets de bord)
    FFT_signal = np.fft.rfft(signal_recu, n=N)
    FFT_motif  = np.fft.rfft(motif_reference, n=N)
    
    # Corrélation = produit dans Fourier (conjugué du motif)
    FFT_correlation = FFT_signal * np.conj(FFT_motif)
    
    # Retour dans le domaine temporel
    correlation = np.fft.irfft(FFT_correlation, n=N)
    
    return correlation[:len(signal_recu)]

# Calculer la corrélation
correlation = correlation_croisee_fft(signal10, motif)

# Trouver le pic de corrélation
position_detectee = np.argmax(np.abs(correlation))
temps_detecte = position_detectee / fs10 * 1000  # en ms
temps_reel = position_vraie / fs10 * 1000         # en ms

print(f'\n📊 Résultats de la détection :')
print(f'   Position réelle du motif  : {temps_reel:.1f} ms')
print(f'   Position détectée         : {temps_detecte:.1f} ms')
print(f'   Erreur de localisation    : {abs(temps_detecte - temps_reel):.1f} ms')
print(f'   → Détection {"✅ RÉUSSIE" if abs(temps_detecte - temps_reel) < 2 else "⚠️ approximative"}')

In [ ]:
# ===== DÉFI 10 : Visualisation =====

fig, axes = plt.subplots(3, 1, figsize=(16, 12))
fig.suptitle('DÉFI 10 — Radar Numérique : Détection par Corrélation Croisée', fontsize=14, fontweight='bold')

t10_ms = t10 * 1000  # Convertir en millisecondes

# Signal reçu (bruité)
axes[0].plot(t10_ms, signal10, color='gray', linewidth=0.5, alpha=0.7, label='Signal reçu (bruité)')
axes[0].axvspan(temps_reel, temps_reel + duree_motif*1000, color='yellow', alpha=0.3, label=f'Motif caché ({temps_reel:.1f}ms)')
axes[0].set_title('Signal reçu — Le motif est INVISIBLE à l\'œil nu (SNR=-10 dB) !')
axes[0].set_xlabel('Temps (ms)'); axes[0].set_ylabel('Amplitude')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Motif de référence
t_motif_ms = np.linspace(0, duree_motif*1000, len(motif))
axes[1].plot(t_motif_ms, motif, color='blue', linewidth=1.5, label='Motif (chirp)')
axes[1].set_title('Motif de référence (signal chirp — ce qu\'on cherche)')
axes[1].set_xlabel('Temps (ms)'); axes[1].set_ylabel('Amplitude')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Corrélation croisée
t_corr_ms = np.linspace(0, duree10*1000, len(correlation))
axes[2].plot(t_corr_ms, np.abs(correlation), color='green', linewidth=0.8, label='|Corrélation|')
axes[2].axvline(temps_detecte, color='red', linewidth=2, linestyle='--', label=f'Pic détecté : {temps_detecte:.1f}ms')
axes[2].axvline(temps_reel, color='blue', linewidth=2, linestyle=':', label=f'Position réelle : {temps_reel:.1f}ms')
axes[2].set_title('Corrélation Croisée — Le pic révèle l\'emplacement exact du motif ! ✅')
axes[2].set_xlabel('Temps (ms)'); axes[2].set_ylabel('Amplitude')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

# Annoter le pic
axes[2].annotate(f'PIC\n{temps_detecte:.1f}ms',
               xy=(temps_detecte, np.abs(correlation[position_detectee])),
               xytext=(temps_detecte + 30, np.abs(correlation[position_detectee]) * 0.8),
               arrowprops=dict(arrowstyle='->', color='red'),
               color='red', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/defi10_radar.png', dpi=150, bbox_inches='tight')
plt.show()

# Gain du matched filter
amplitude_bruit_corr = np.std(np.abs(correlation))
amplitude_pic_corr = np.abs(correlation[position_detectee])
gain_db = 20 * np.log10(amplitude_pic_corr / amplitude_bruit_corr)

print(f'\n📊 Gain du Matched Filter : {gain_db:.1f} dB')
print(f'   → La corrélation amplifie le signal de {gain_db:.1f} dB !')
print(f'   → Même avec un SNR = {snr_cible_db} dB, la détection est réussie !')
print('✅ Défi 10 terminé !')

---
# 📋 BILAN FINAL — Récapitulatif de tous les défis

Ce notebook a couvert **11 défis** de traitement du signal, du son et de l'image.

In [ ]:
# ===== BILAN FINAL =====

print('=' * 70)
print('📋 BILAN FINAL — Mini-Projets TMA 2025-2026 — SupNum')
print('=' * 70)

bilan = [
    ('🔴 OBLIGATOIRE', 'Analyse Spectrale de Voix',     'FFT, Pitch, Harmoniques, Spectrogramme'),
    ('🟠 Défi 1',     'Archéologue Acoustique',         'Filtre Notch, Soustraction Spectrale, SNR'),
    ('🟡 Défi 2',     'Séparation de Sources (BSS)',    'SVD, ICA (FastICA), Blanchiment'),
    ('🟢 Défi 3',     'Détective de l\'Aliasing',       'Shannon, Repliement spectral, Interpolation sinc'),
    ('🔵 Défi 4',     'Diagnostic du Futur (ECG)',      'Filtre Butterworth, Détection QRS, BPM'),
    ('🟣 Défi 5',     'Opticien Numérique (Déconv.)',   'PSF, Convolution 2D, Filtre de Wiener, PSNR'),
    ('🩷 Défi 6',     'Chasseur de Fantômes (ISO)',      'FFT 2D, Masque fréquentiel, Filtre Médian'),
    ('🤍 Défi 7',     'Espion Fréquentiel (Stégano.)',  'LSB, Encodage binaire, Invisibilité'),
    ('🖤 Défi 8',     'Compresseur Intelligent (MP3)',  'STFT, Masquage psychoacoustique, Compression'),
    ('🟫 Défi 9',     'Annulateur d\'Écho (AEC)',       'Réponse impulsionnelle, LMS, Filtrage adaptatif'),
    ('🔶 Défi 10',    'Radar Numérique',                'Corrélation croisée, FFT rapide, Matched Filter'),
]

print(f'\n{"Défi":<18} {"Titre":<32} {"Concepts clés"}')
print('-' * 100)
for emoji, titre, concepts in bilan:
    print(f'{emoji:<18} {titre:<32} {concepts}')

print('\n' + '=' * 70)
print('✅ Tous les défis ont été complétés !')
print('\n📌 OUTILS UTILISÉS :')
print('   numpy         → Calcul numérique, FFT')
print('   scipy.signal  → Filtres (Notch, Butterworth), Corrélation, STFT')
print('   matplotlib    → Visualisations, Spectrogrammes')
print('   skimage       → Traitement d\'images (données test)')
print('   scipy.fft     → FFT rapide 1D et 2D')
print('=' * 70)

In [ ]:
# ===== Lister tous les fichiers générés =====
import os
import glob

print('📁 Fichiers générés dans /content/ :')
fichiers = sorted(glob.glob('/content/defi*.png'))
for f in fichiers:
    taille = os.path.getsize(f) / 1024
    print(f'   {os.path.basename(f):40s} {taille:.1f} Ko')

print(f'\nTotal : {len(fichiers)} graphiques générés')
print('\n💡 Pour télécharger les fichiers : Cliquez sur l\'icône de dossier (gauche) → /content/')

---
## 📚 Sources et Datasets utilisés

| Défi | Dataset / Source |
|------|------------------|
| Obligatoire | Signal simulé (numpy) — représente une voix humaine |
| Défi 1 | Signal synthétique simulant le discours JFK |
| Défi 2 | Signaux synthétiques simulant le Cocktail Party Problem |
| Défi 3 | Signal chirp généré par `scipy.signal.chirp` |
| Défi 4 | ECG synthétique (ondes P, QRS, T) |
| Défi 5 | `skimage.data.astronaut()` — image astronaute |
| Défi 6 | `skimage.data.coins()` — image pièces de monnaie |
| Défi 7 | `skimage.data.astronaut()` — image hôte |
| Défi 8 | Signal audio synthétique multi-fréquences |
| Défi 9 | Voix et haut-parleur simulés |
| Défi 10 | Signal chirp radar + bruit gaussien |

**Bibliothèques :** `numpy`, `scipy`, `matplotlib`, `scikit-image`, `Pillow`

---
*Mini-Projet TMA 2025-2026 — Institut Supérieur du Numérique (SupNum)*  
*Dr. EL BENANY Med Mahmoud*